In [1]:
!pip install rdkit
!pip install pandas numpy scikit-learn -U
!pip install pytdc --no-dependencies
!pip install fuzzywuzzy

  Using cached fuzzywuzzy-0.18.0-py2.py3-none-any.whl.metadata (4.9 kB)
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytdc 1.1.15 requires cellxgene-census==1.15.0, which is not installed.
pytdc 1.1.15 requires dataclasses<1.0,>=0.6, which is not installed.
pytdc 1.1.15 requires evaluate==0.4.2, which is not installed.
pytdc 1.1.15 requires gget<1.0.0,>=0.28.4, which is not installed.
pytdc 1.1.15 requires tiledbsoma<2.0.0,>=1.7.2, which is not installed.
pytdc 1.1.15 requires accelerate==0.33.0, but you have accelerate 1.13.0 which is incompatible.
pytdc 1.1.15 requires datasets<2.20.0, but you have datasets 4.0.0 which is incompatible.
pytdc 1.1.15 requires huggingface-hub<1.0,>=0.20.3, but you have huggingface-hub 1.10.1 which is incompatible.
pytdc 1.1.15 requires numpy<2.0.0,>=1.26.4, but you have numpy 2.4.4 which is incompatible.
pytdc 1.1.15 requires

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [18]:
data_folder = "/content/drive/MyDrive/mldd_data"

In [19]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
from tdc.single_pred import ADME
from rdkit import Chem
from rdkit.Chem import Descriptors, Lipinski
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score

# ==========================================
# 1. DEFINICJE KLAS (Architektura i Featurizer)
# ==========================================

class AdmetEncoder(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()
        self.main = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout)
        )
        self.res_layer = nn.Linear(hidden_dim, hidden_dim)

    def forward(self, x):
        x = self.main(x)
        return torch.relu(x + self.res_layer(x))

# --- STL REGRESOR ---
class STL_NN_Regressor(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1) # Wynik liniowy (dowolna liczba)

    def forward(self, x):
        return self.head(self.encoder(x))

# --- STL KLASYFIKATOR ---
class STL_NN_Classifier(nn.Module):
    def __init__(self, input_dim, hidden_dim=512):
        super().__init__()
        self.encoder = AdmetEncoder(input_dim, hidden_dim)
        self.head = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # Sigmoid zamienia wynik na prawdopodobieństwo (0-1)
        return torch.sigmoid(self.head(self.encoder(x)))

def load_embeddings(dataset_name, train_df, test_df):
    """Wczytuje CSV z embeddingami i dzieli na train/test po SMILES (tak samo jak featurizer)."""
    file_path = f"{data_folder}/{dataset_name}_MoLFormer_embeddings.csv"
    emb_df = pd.read_csv(file_path)

    train_smiles = set(train_df['Drug'].tolist())
    test_smiles  = set(test_df['Drug'].tolist())

    emb_train = emb_df[emb_df['Drug'].isin(train_smiles)]
    emb_test  = emb_df[emb_df['Drug'].isin(test_smiles)]

    feature_cols = [c for c in emb_df.columns if c.startswith('emb_')]

    X_train = emb_train[feature_cols].values
    y_train = emb_train['Y'].values
    X_test  = emb_test[feature_cols].values
    y_test  = emb_test['Y'].values

    return X_train, y_train, X_test, y_test

In [20]:
import pickle

def load_split_pickle(dataset_name):
    filepath = f"{data_folder}/{dataset_name}_split.pkl"

    with open(filepath, "rb") as f:
        split = pickle.load(f)

    return split["train"], split["test"]

In [21]:
def print_metrics(metrics, task='classification'):
    print(f"\n{'='*40}")
    if task == 'classification':
        print(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}")
        print(f"  F1       : {metrics['test_metrics']['f1']:.4f}")
        print(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}")
    else:
        print(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}")
        print(f"  MAE      : {metrics['test_metrics']['mae']:.4f}")
        print(f"  R²       : {metrics['test_metrics']['r2']:.4f}")
    print(f"{'='*40}\n")


def save_metrics(metrics, dataset_name, filepath, task='classification'):
    with open(filepath, 'a') as f:
        f.write(f"\n{'='*40}\n")
        f.write(f"Endpoint    : {dataset_name}\n")
        f.write(f"{'='*40}\n")
        if task == 'classification':
            f.write(f"  Accuracy : {metrics['test_metrics']['accuracy']:.4f}\n")
            f.write(f"  F1       : {metrics['test_metrics']['f1']:.4f}\n")
            f.write(f"  AUROC    : {metrics['test_metrics']['auroc']:.4f}\n")
        else:
            f.write(f"  RMSE     : {metrics['test_metrics']['rmse']:.4f}\n")
            f.write(f"  MAE      : {metrics['test_metrics']['mae']:.4f}\n")
            f.write(f"  R²       : {metrics['test_metrics']['r2']:.4f}\n")
        f.write(f"{'='*40}\n")

In [30]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

def train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np):

    # C. Normalizacja etykiet
    print("[INFO] Normalizacja danych wyjściowych...")
    scaler = StandardScaler()
    y_train_scaled = scaler.fit_transform(y_train_np.reshape(-1, 1))
    y_test_scaled = scaler.transform(y_test_np.reshape(-1, 1))

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_scaled).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

    model = STL_NN_Regressor(input_dim=X_train_np.shape[1]).to(device)
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    print("\n--- START TRENINGU ---")
    model.train()
    for epoch in range(100):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            mse_val = loss.item()
            print(f"  Epoka {epoch:3d} | Błąd MSE: {loss.item():.4f}")

# F. Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        preds_scaled = model(X_test_t).cpu().numpy()

    preds = scaler.inverse_transform(preds_scaled)

    mse = mean_squared_error(y_test_np, preds)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test_np, preds)
    r2 = r2_score(y_test_np, preds)

    metrics = {
        "test_metrics": {
            "rmse": rmse,
            "mae": mae,
            "r2": r2
        }
    }

    return metrics

In [31]:
from sklearn.metrics import accuracy_score, roc_auc_score, f1_score
def train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    X_train_t = torch.FloatTensor(X_train_np).to(device)
    y_train_t = torch.FloatTensor(y_train_np.reshape(-1, 1)).to(device)
    X_test_t = torch.FloatTensor(X_test_np).to(device)

# D. Model i Trening
    model = STL_NN_Classifier(input_dim=X_train_np.shape[1]).to(device)
    criterion = nn.BCELoss() # Binary Cross Entropy - standard dla klasyfikacji
    optimizer = optim.Adam(model.parameters(), lr=0.0005)

    print("\n--- START TRENINGU (Klasyfikacja) ---")
    model.train()
    for epoch in range(101):
        optimizer.zero_grad()
        outputs = model(X_train_t)
        loss = criterion(outputs, y_train_t)
        loss.backward()
        optimizer.step()

        if epoch % 20 == 0:
            print(f"  Epoka {epoch:3d} | Błąd BCE Loss: {loss.item():.4f}")

# E. Ewaluacja
    print("\n--- EWALUACJA ---")
    model.eval()
    with torch.no_grad():
        # Model zwraca prawdopodobieństwo (0 do 1)
        probs = model(X_test_t).cpu().numpy()
        # Klasy (0 lub 1) ustalamy progiem 0.5
    preds = (probs > 0.5).astype(int)

    # Reshape y_test_np to match the (N, 1) shape of probs/preds for metrics
    y_test_np_reshaped = y_test_np.reshape(-1, 1)

    # Obliczamy AUROC - kluczowa metryka dla klasyfikacji ADMET
    auroc = roc_auc_score(y_test_np_reshaped, probs)
    acc = accuracy_score(y_test_np_reshaped, preds)

    print(f"Wynik końcowy AUROC (HIA): {auroc:.4f}")
    print(f"Dokładność (Accuracy): {acc*100:.2f}%")

    metrics = {
        "test_metrics": {
            "accuracy": accuracy_score(y_test_np_reshaped, preds),
            "f1":       f1_score(y_test_np_reshaped, preds),
            "auroc":    roc_auc_score(y_test_np_reshaped, probs)
        }
    }

    return metrics

In [32]:
embeddings_folder = '/content/drive/MyDrive/MLDD - ADMET/STL_ML'

Endpoint 1: Caco-2 (Wang)

In [33]:
train, test = load_split_pickle('Caco2_Wang')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('Caco2_Wang', train, test)


metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Caco2_Wang", "metrics_NN_embeddings.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1629
  Epoka  20 | Błąd MSE: 0.3647
  Epoka  40 | Błąd MSE: 0.2246
  Epoka  60 | Błąd MSE: 0.1293
  Epoka  80 | Błąd MSE: 0.0730

--- EWALUACJA ---

  RMSE     : 0.4768
  MAE      : 0.3720
  R²       : 0.6442



Endpoint 2:  Lipophilicity

In [34]:
train, test = load_split_pickle('Lipophilicity_AstraZeneca')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('Lipophilicity_AstraZeneca', train, test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Lipophilicity_AstraZeneca", "metrics_NN_embeddings.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.0321
  Epoka  20 | Błąd MSE: 0.7339
  Epoka  40 | Błąd MSE: 0.5553
  Epoka  60 | Błąd MSE: 0.4049
  Epoka  80 | Błąd MSE: 0.2731

--- EWALUACJA ---

  RMSE     : 0.7370
  MAE      : 0.5725
  R²       : 0.6323



Endpoint 3: Solubility

In [35]:
train, test = load_split_pickle('Solubility_AqSolDB')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('AqSolDB', train, test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Solubility_AqSolDB", "metrics_NN_embeddings.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1694
  Epoka  20 | Błąd MSE: 0.3884
  Epoka  40 | Błąd MSE: 0.3024
  Epoka  60 | Błąd MSE: 0.2446
  Epoka  80 | Błąd MSE: 0.1983

--- EWALUACJA ---

  RMSE     : 1.1068
  MAE      : 0.8094
  R²       : 0.7743



Endpoint 4: HIA

In [36]:
train, test = load_split_pickle('HIA_Hou')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('HIA_Hou', train, test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "HIA_Hou", "metrics_NN_embeddings.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.6584
  Epoka  20 | Błąd BCE Loss: 0.0885
  Epoka  40 | Błąd BCE Loss: 0.0052
  Epoka  60 | Błąd BCE Loss: 0.0008
  Epoka  80 | Błąd BCE Loss: 0.0003
  Epoka 100 | Błąd BCE Loss: 0.0002

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.9881
Dokładność (Accuracy): 95.69%

  Accuracy : 0.9569
  F1       : 0.9741
  AUROC    : 0.9881



Endpoint 5: Half Life

In [37]:
train, test = load_split_pickle('Half_Life_Obach')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('Half_Life_Obach', train, test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Half_Life_Obach", "metrics_NN_embeddings.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1694
  Epoka  20 | Błąd MSE: 0.5571
  Epoka  40 | Błąd MSE: 0.2691
  Epoka  60 | Błąd MSE: 0.0807
  Epoka  80 | Błąd MSE: 0.0290

--- EWALUACJA ---

  RMSE     : 93.5280
  MAE      : 26.9926
  R²       : 0.3856



Endpoint 6: Clearance Hepatocyte

In [38]:
train, test = load_split_pickle('Clearance_Hepatocyte_AZ')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('Clearance_Hepatocyte_AZ', train, test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "Clearance_Hepatocyte_AZ", "metrics_NN_embeddings.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.1432
  Epoka  20 | Błąd MSE: 0.7308
  Epoka  40 | Błąd MSE: 0.4974
  Epoka  60 | Błąd MSE: 0.2477
  Epoka  80 | Błąd MSE: 0.1246

--- EWALUACJA ---

  RMSE     : 36.5755
  MAE      : 23.1591
  R²       : 0.4474



Endpoint 7: CYP3A4 Inhibition

In [39]:
train, test = load_split_pickle('CYP3A4_Veith')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('CYP3A4_Veith', train, test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "CYP3A4_Veith", "metrics_NN_embeddings.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.7387
  Epoka  20 | Błąd BCE Loss: 0.4816
  Epoka  40 | Błąd BCE Loss: 0.4054
  Epoka  60 | Błąd BCE Loss: 0.3144
  Epoka  80 | Błąd BCE Loss: 0.2075
  Epoka 100 | Błąd BCE Loss: 0.1280

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.8788
Dokładność (Accuracy): 80.01%

  Accuracy : 0.8001
  F1       : 0.7509
  AUROC    : 0.8788



Endpoint 8: VDss

In [40]:
train, test = load_split_pickle('VDss_Lombardo')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('VDss_Lombardo', train, test)

metrics = train_NN_regression(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics, task="regression")

save_metrics(metrics, "VDss_Lombardo", "metrics_NN_embeddings.txt", task="regression")

[INFO] Normalizacja danych wyjściowych...

--- START TRENINGU ---
  Epoka   0 | Błąd MSE: 1.2326
  Epoka  20 | Błąd MSE: 1.0139
  Epoka  40 | Błąd MSE: 0.7873
  Epoka  60 | Błąd MSE: 0.5234
  Epoka  80 | Błąd MSE: 0.3238

--- EWALUACJA ---

  RMSE     : 7.5154
  MAE      : 3.9647
  R²       : -0.1870



Endpoint 9: AMES Mutagenicity

In [41]:
train, test = load_split_pickle('AMES')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('AMES', train, test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "AMES", "metrics_NN_embeddings.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.7228
  Epoka  20 | Błąd BCE Loss: 0.5200
  Epoka  40 | Błąd BCE Loss: 0.4105
  Epoka  60 | Błąd BCE Loss: 0.2761
  Epoka  80 | Błąd BCE Loss: 0.1549
  Epoka 100 | Błąd BCE Loss: 0.0805

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.8812
Dokładność (Accuracy): 80.61%

  Accuracy : 0.8061
  F1       : 0.8260
  AUROC    : 0.8812



Endpoint 10: hERG (Wang)

In [42]:
train, test = load_split_pickle('hERG')

X_train_np, y_train_np, X_test_np, y_test_np = load_embeddings('hERG', train, test)

metrics = train_NN_classification(X_train_np, X_test_np, y_train_np, y_test_np)

print_metrics(metrics)

save_metrics(metrics, "hERG", "metrics_NN_embeddings.txt")


--- START TRENINGU (Klasyfikacja) ---
  Epoka   0 | Błąd BCE Loss: 0.7707
  Epoka  20 | Błąd BCE Loss: 0.2506
  Epoka  40 | Błąd BCE Loss: 0.0386
  Epoka  60 | Błąd BCE Loss: 0.0029
  Epoka  80 | Błąd BCE Loss: 0.0005
  Epoka 100 | Błąd BCE Loss: 0.0003

--- EWALUACJA ---
Wynik końcowy AUROC (HIA): 0.8692
Dokładność (Accuracy): 84.96%

  Accuracy : 0.8496
  F1       : 0.9010
  AUROC    : 0.8692

